<a href="https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/notebooks/L4S_09_loro_cross_region.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L4S_09 — LORO Cross-Region · Generalización Geográfica · U-Net + ResNet-34
**Proyecto:** Detección de Deslizamientos — Landslide4Sense  
**Protocolo:** Leave-One-Region-Out (LORO) — 4 regiones geográficas  
**Modelo:** U-Net con encoder ResNet-34 (`segmentation-models-pytorch`)  
**Objetivo:** Medir cuánto pierde el modelo al ser evaluado en una región geográficamente distinta a su entrenamiento — puente clave hacia el contexto colombiano.

---

## ¿Qué es LORO y por qué es importante?

La validación cruzada 5-fold estándar (L4S_04) mezcla parches de todas las regiones en train y validación, estimando el rendimiento **dentro de la misma distribución geográfica**. Esto sobreestima el desempeño cuando el modelo se aplica a una región completamente nueva.

LORO simula exactamente ese escenario: entrena con 3 regiones (≈2,300–3,350 parches) y evalúa en la 4ª región no vista. El **gap LORO − 5-Fold** cuantifica la degradación por cambio de dominio geográfico.

| Protocolo | Qué mide | Relevancia Colombia |
|-----------|----------|--------------------|
| 5-Fold CV | Rendimiento en distribución conocida | Referencia de laboratorio |
| **LORO** | **Generalización a región nueva** | **Proxy directo de transferencia** |

## Regiones del dataset L4S (3 799 parches)

| ID | Región | País | Evento | Detonante | Parches | Índices |
|----|--------|------|--------|-----------|---------|--------|
| 0 | Iburi | Japón | Terremoto 2018 | Sísmico | 1 000 | 0–999 |
| 1 | Kodagu | India | Monzón 2018 | Lluvia intensa | 450 | 1000–1449 |
| 2 | Gorkha | Nepal | Terremoto 2015 | Sísmico | 850 | 1450–2299 |
| 3 | Taiwan | Taiwán | Tifón 2009 | Lluvia + viento | 1 499 | 2300–3798 |

> **Nota:** Los índices asumen ordenamiento numérico de `image_1.h5 … image_3799.h5`. Esta distribución es la reportada en el paper ISPRS Landslide4Sense (Ghorbanzadeh et al., 2022, IEEE TGRS). Si tu dataset tiene una ordenación diferente, ajusta `REGION_RANGES` en la Celda 3.

## Flujo de reanudación
```
Sesión 1:  Región 0 (Iburi) ✅  → región0_iburi_results.json guardado en Drive
Sesión 2:  Región 1 (Kodagu) ✅ → región1_kodagu_results.json
...
Sesión N:  Celda 8 → resumen LORO vs 5-Fold + análisis de gap
```
> ⏱ **Tiempo estimado en T4:** ~30–45 min por región → ~2–3 h total.

In [ ]:
# ── Celda 1: Instalación, Drive y carga de datos ───────────────────────────
from google.colab import drive
import os, sys, h5py, json, subprocess, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from tqdm.auto import tqdm

for pkg in ['segmentation-models-pytorch', 'timm', 'h5py', 'tqdm']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

drive.mount('/content/drive')

base_path = Path('/content/drive/MyDrive/Landslide4Sense')
img_dirs  = list(base_path.glob('**/TrainData/img'))
if not img_dirs:
    raise FileNotFoundError('❌ No se encontró TrainData en Drive. '
                            'Verifica que el dataset esté en /content/drive/MyDrive/Landslide4Sense/')

train_img_dir  = img_dirs[0]
train_mask_dir = train_img_dir.parent / 'mask'

# Ordenar numéricamente: image_1.h5, image_2.h5, ..., image_3799.h5
img_list  = sorted(list(train_img_dir.glob('*.h5')),
                   key=lambda p: int(p.stem.split('_')[-1]))
mask_list = [train_mask_dir / p.name.replace('image_', 'mask_') for p in img_list]

# Verificar que las máscaras existen
missing = [m for m in mask_list if not m.exists()]
if missing:
    print(f'⚠️  {len(missing)} máscaras no encontradas (p.ej. {missing[0].name})')
    print('   Intentando con mismo nombre que imagen...')
    mask_list = [train_mask_dir / p.name for p in img_list]
    missing2 = [m for m in mask_list if not m.exists()]
    if missing2:
        raise FileNotFoundError(f'❌ No se encontraron máscaras en {train_mask_dir}')

img_arr  = np.array(img_list)
mask_arr = np.array(mask_list)
print(f'✅ {len(img_list)} parches | ordenados: {img_list[0].name} … {img_list[-1].name}')
print(f'   Train img : {train_img_dir}')
print(f'   Train mask: {train_mask_dir}')

In [ ]:
# ── Celda 2: Asignación de regiones geográficas ────────────────────────────
#
# Distribución reportada en Ghorbanzadeh et al. 2022 (IEEE TGRS):
#   Iburi/Japón:   1 000 parches  →  índices  0 – 999
#   Kodagu/India:    450 parches  →  índices  1000 – 1449
#   Gorkha/Nepal:    850 parches  →  índices  1450 – 2299
#   Taiwan:        1 499 parches  →  índices  2300 – 3798
#   TOTAL:         3 799 parches
#
# Si tu copia del dataset tiene un orden diferente, edita REGION_RANGES aquí.

REGION_RANGES = {
    0: {'name': 'Iburi',  'country': 'Japón',   'trigger': 'Sísmico',     'start': 0,    'end': 999},
    1: {'name': 'Kodagu', 'country': 'India',   'trigger': 'Lluvia',      'start': 1000, 'end': 1449},
    2: {'name': 'Gorkha', 'country': 'Nepal',   'trigger': 'Sísmico',     'start': 1450, 'end': 2299},
    3: {'name': 'Taiwan', 'country': 'Taiwán',  'trigger': 'Tifón/Lluvia','start': 2300, 'end': 3798},
}

N_TOTAL = len(img_arr)
assert N_TOTAL == 3799, f'Se esperaban 3799 parches, encontrados: {N_TOTAL}'

# Construir array de etiquetas de región (0-3) por parche
region_labels = np.zeros(N_TOTAL, dtype=np.int32)
for rid, info in REGION_RANGES.items():
    region_labels[info['start']:info['end']+1] = rid

# Construir caché de etiquetas de parche (positivo = tiene deslizamiento)
cache_path = base_path / 'results' / 'labels_cache.json'
if cache_path.exists():
    with open(cache_path) as f:
        cache = json.load(f)
    all_labels = np.array(cache['labels'], dtype=np.int32)
    print(f'✅ Caché de etiquetas cargado ({len(all_labels)} parches)')
else:
    print('⏳ Generando caché de etiquetas (puede tardar ~3 min)...')
    all_labels = []
    for mp in tqdm(mask_list, desc='Leyendo máscaras'):
        with h5py.File(mp, 'r') as f:
            key = list(f.keys())[0]
            mask = f[key][()]
        all_labels.append(int(mask.max() > 0))
    all_labels = np.array(all_labels, dtype=np.int32)
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    with open(cache_path, 'w') as f:
        json.dump({'labels': all_labels.tolist(), 'n': len(all_labels)}, f)
    print(f'✅ Caché guardado en {cache_path}')

# Resumen por región
print()
print('DISTRIBUCIÓN POR REGIÓN')
print('='*65)
print(f'{"ID":>3} {"Región":10} {"País":8} {"Parches":>8} {"Positivos":>10} {"% pos":>7} {"Índices"}')
print('-'*65)
for rid, info in REGION_RANGES.items():
    mask_r   = region_labels == rid
    n_tot    = mask_r.sum()
    n_pos    = all_labels[mask_r].sum()
    pct      = 100 * n_pos / n_tot
    print(f'{rid:>3} {info["name"]:10} {info["country"]:8} {n_tot:>8} {n_pos:>10} {pct:>7.1f}%  '
          f'{info["start"]}–{info["end"]}')
print('='*65)
print(f'    {"TOTAL":10} {"":8} {N_TOTAL:>8} {all_labels.sum():>10} '
      f'{100*all_labels.mean():>7.1f}%')

In [ ]:
# ── Celda 3: Modelo U-Net + ResNet-34 ──────────────────────────────────────
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp

print(f'PyTorch: {torch.__version__} | smp: {smp.__version__} | CUDA: {torch.cuda.is_available()}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cpu':
    print('⚠️  Sin GPU — el entrenamiento será muy lento. Activa Runtime > Cambiar tipo > T4.')

def build_unet(n_channels=14, pretrained=True):
    """U-Net con encoder ResNet-34 preentrenado en ImageNet.
    Los 14 canales de entrada se adaptan mediante proyección 1x1 sobre los 3 canales RGB."""
    model = smp.Unet(
        encoder_name    = 'resnet34',
        encoder_weights = 'imagenet' if pretrained else None,
        in_channels     = n_channels,
        classes         = 1,
    )
    return model

# Verificar que el modelo compila
_m = build_unet(14).to(device)
_x = torch.zeros(2, 14, 128, 128).to(device)
with torch.no_grad():
    _y = _m(_x)
print(f'✅ U-Net output shape: {_y.shape}  (esperado: [2, 1, 128, 128])')
del _m, _x, _y
torch.cuda.empty_cache()

In [ ]:
# ── Celda 4: Loss DiceBCE + Dataset de Segmentación ────────────────────────
from torch.utils.data import Dataset, DataLoader

class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__(); self.smooth = smooth

    def forward(self, logits, targets):
        probs  = torch.sigmoid(logits)
        flat_p = probs.view(-1); flat_t = targets.view(-1)
        return 1 - (2*(flat_p*flat_t).sum() + self.smooth) / (flat_p.sum() + flat_t.sum() + self.smooth)

class DiceBCELoss(nn.Module):
    """Loss combinado para segmentación con desbalance de clases."""
    def __init__(self, pos_weight=None, alpha=0.5):
        super().__init__()
        self.dice  = DiceLoss()
        self.bce   = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        self.alpha = alpha

    def forward(self, logits, targets):
        return self.alpha * self.dice(logits, targets) + (1-self.alpha) * self.bce(logits, targets)


class SegmentationDataset(Dataset):
    def __init__(self, img_paths, mask_paths, augment=False):
        self.img_paths  = img_paths
        self.mask_paths = mask_paths
        self.augment    = augment

    def __len__(self): return len(self.img_paths)

    def __getitem__(self, idx):
        with h5py.File(self.img_paths[idx], 'r') as f:
            key = list(f.keys())[0]
            img = f[key][()].astype(np.float32)   # (128,128,14)
        with h5py.File(self.mask_paths[idx], 'r') as f:
            key = list(f.keys())[0]
            mask = f[key][()].astype(np.float32)  # (128,128)

        # Clip + Z-score por canal
        for c in range(img.shape[2]):
            ch = img[:, :, c]
            p1, p99 = np.percentile(ch, [1, 99])
            ch = np.clip(ch, p1, p99)
            img[:, :, c] = (ch - ch.mean()) / (ch.std() + 1e-8)

        img_t  = torch.from_numpy(img.transpose(2, 0, 1))   # (14,128,128)
        mask_t = torch.from_numpy(mask).unsqueeze(0)        # (1,128,128)

        if self.augment:
            # Flips
            if torch.rand(1).item() > 0.5:
                img_t  = torch.flip(img_t,  dims=[2])
                mask_t = torch.flip(mask_t, dims=[2])
            if torch.rand(1).item() > 0.5:
                img_t  = torch.flip(img_t,  dims=[1])
                mask_t = torch.flip(mask_t, dims=[1])
            # Rotaciones 90°/180°/270°
            k = torch.randint(0, 4, (1,)).item()
            if k > 0:
                img_t  = torch.rot90(img_t,  k, dims=[1, 2])
                mask_t = torch.rot90(mask_t, k, dims=[1, 2])

        return img_t, mask_t

print('✅ DiceBCELoss y SegmentationDataset definidos (con augmentation completa: flips + rot90)')

## Entrenamiento LORO — 4 regiones × 20 épocas

| Parámetro | Valor | Justificación |
|-----------|-------|---------------|
| `epochs` | 20 | Mismo que L4S_04 para comparabilidad |
| `patience` | 5 | Early stopping |
| `batch_size` | 16 | VRAM T4 |
| `lr` | 1e-3 | OneCycleLR |
| Augmentación | H-flip + V-flip + rot90 | Misma que L4S_04 |

### 🔄 Control de sesión
- `REGION_TO_RUN = None` → corre todas las regiones pendientes  
- `REGION_TO_RUN = 0/1/2/3` → corre solo esa región  
- Si `regionN_results.json` ya existe en Drive → se salta automáticamente

In [ ]:
# ── Celda 5: Configuración LORO y estado de regiones ──────────────────────
#
# ┌─────────────────────────────────────────────────────────────────────────┐
# │  REGION_TO_RUN  →  None = correr todas las pendientes                   │
# │                 →  0/1/2/3 = correr SOLO esa región                     │
# └─────────────────────────────────────────────────────────────────────────┘
REGION_TO_RUN = None   # <── ajusta aquí antes de correr

CFG = {
    'epochs'     : 20,
    'batch_size' : 16,
    'lr'         : 1e-3,
    'pos_weight' : 0.703,   # < 1.0 porque dentro de parches positivos los píxeles pos siguen siendo minoría
    'patience'   : 5,
    'seed'       : 42,
    'use_amp'    : True,
    'thr_pixel'  : 0.5,
}

torch.manual_seed(CFG['seed']); np.random.seed(CFG['seed'])

output_dir = base_path / 'results' / 'comparable_literatura' / 'loro_cross_region'
output_dir.mkdir(parents=True, exist_ok=True)

# ── Helpers de estado ──────────────────────────────────────────────────────
def region_is_done(rid):
    """True si la región ya tiene checkpoint Y resultados en Drive."""
    name = REGION_RANGES[rid]['name'].lower()
    ckpt = output_dir / f'region{rid}_{name}_best.pt'
    res  = output_dir / f'region{rid}_{name}_results.json'
    return ckpt.exists() and res.exists()

def load_region_results(rid):
    name = REGION_RANGES[rid]['name'].lower()
    with open(output_dir / f'region{rid}_{name}_results.json') as f:
        return json.load(f)

# ── Resumen de estado ──────────────────────────────────────────────────────
done    = [rid for rid in range(4) if region_is_done(rid)]
pending = [rid for rid in range(4) if not region_is_done(rid)]

print('='*65)
print('  ESTADO LORO')
print('='*65)
for rid, info in REGION_RANGES.items():
    n_train = sum(
        REGION_RANGES[r]['end'] - REGION_RANGES[r]['start'] + 1
        for r in range(4) if r != rid
    )
    n_test  = info['end'] - info['start'] + 1
    status  = '✅ completado' if region_is_done(rid) else '⏳ pendiente'
    print(f'  Región {rid} ({info["name"]:6}, {info["country"]:6}): '
          f'train={n_train:4d}  test={n_test:4d}  {status}')
print('='*65)

if REGION_TO_RUN is not None:
    print(f'\n🎯 Modo sesión: correr SOLO Región {REGION_TO_RUN} '
          f'({REGION_RANGES[REGION_TO_RUN]["name"]})')
    if region_is_done(REGION_TO_RUN):
        r = load_region_results(REGION_TO_RUN)
        print(f'   ⚠️  Ya completada — F1={r["f1_pixel_thr05"]:.4f}')
        print(f'      Para re-entrenar, borra region{REGION_TO_RUN}_*  en Drive.')
else:
    print(f'\n▶️  Se correrán las regiones pendientes: {pending}')
    if not pending:
        print('   🎉 ¡Todas las regiones completadas! Ve a la Celda 7 para resultados.')

In [ ]:
# ── Celda 6: Entrenamiento LORO ────────────────────────────────────────────
#
# Por cada región held-out:
#   regionN_{name}_best.pt         → mejor checkpoint
#   regionN_{name}_results.json    → métricas completas
#   regionN_{name}_vis.npz         → muestras visuales (4 parches del test)
#
import torch.optim as optim
from torch.amp import GradScaler, autocast
from sklearn.metrics import (f1_score, roc_auc_score, jaccard_score,
                              average_precision_score, precision_recall_curve)

regions_to_run = ([REGION_TO_RUN] if REGION_TO_RUN is not None
                  else [r for r in range(4) if not region_is_done(r)])

if not regions_to_run:
    print('🎉 No hay regiones pendientes. Ejecuta la Celda 7 para el resumen.')
else:
    print(f'▶️  Regiones a entrenar en esta sesión: '
          f'{[REGION_RANGES[r]["name"] for r in regions_to_run]}\n')

for rid in regions_to_run:
    info = REGION_RANGES[rid]
    name = info['name'].lower()

    if region_is_done(rid):
        r = load_region_results(rid)
        print(f'⏭  {info["name"]} ({info["country"]}) ya completada — '
              f'F1={r["f1_pixel_thr05"]:.4f} | Dice={r["dice_thr05"]:.4f} | '
              f'IoU={r["iou_thr05"]:.4f} — skip')
        continue

    # ── Splits LORO ─────────────────────────────────────────────────────────
    test_idx  = np.where(region_labels == rid)[0]
    train_idx = np.where(region_labels != rid)[0]
    train_regions = [REGION_RANGES[r]['name'] for r in range(4) if r != rid]

    # Calcular pos_weight local para el set de entrenamiento
    train_labels = all_labels[train_idx]
    n_neg_train  = (train_labels == 0).sum()
    n_pos_train  = (train_labels == 1).sum()
    local_pos_w  = CFG['pos_weight']  # usar el valor global calibrado

    print(f'\n{"="*65}')
    print(f'LORO — Held-out: {info["name"]} ({info["country"]}) | Detonante: {info["trigger"]}')
    print(f'  Train: {len(train_idx):4d} parches de {train_regions}')
    print(f'  Test:  {len(test_idx):4d} parches de {info["name"]}')
    print(f'  Train pos/neg: {n_pos_train}/{n_neg_train} '
          f'({100*n_pos_train/len(train_idx):.1f}% positivos)')
    print('='*65)

    model = build_unet(n_channels=14, pretrained=True).to(device)

    train_ds = SegmentationDataset(img_arr[train_idx].tolist(), mask_arr[train_idx].tolist(), augment=True)
    test_ds  = SegmentationDataset(img_arr[test_idx].tolist(),  mask_arr[test_idx].tolist(),  augment=False)
    train_dl = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,  num_workers=2, pin_memory=True)
    test_dl  = DataLoader(test_ds,  batch_size=CFG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)

    pos_w     = torch.tensor([local_pos_w], device=device)
    criterion = DiceBCELoss(pos_weight=pos_w)
    optimizer = optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=1e-4)
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=CFG['lr'],
        steps_per_epoch=len(train_dl), epochs=CFG['epochs'])
    scaler = GradScaler('cuda', enabled=CFG['use_amp'])

    history = {'train_loss': [], 'val_loss': [], 'val_f1_pixel': [], 'val_dice': [], 'val_iou': []}
    best_f1, best_epoch, no_improve = 0.0, 0, 0
    t_region = time.time()

    for epoch in range(1, CFG['epochs'] + 1):
        # ── Train ────────────────────────────────────────────────────────────
        model.train()
        train_loss = 0.0
        for imgs, masks in train_dl:
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad()
            with autocast('cuda', enabled=CFG['use_amp']):
                logits = model(imgs); loss = criterion(logits, masks)
            scaler.scale(loss).backward()
            scaler.step(optimizer); scaler.update(); scheduler.step()
            train_loss += loss.item()
        train_loss /= len(train_dl)

        # ── Evaluación en región test (held-out) ─────────────────────────────
        # NOTA: En LORO el 'test' es la región held-out — no hay val separado durante
        # entrenamiento. Usamos la misma región test para early stopping (proxy).
        # Esto es la práctica estándar cuando no hay un 4º split disponible.
        model.eval()
        test_loss = 0.0
        all_probs_flat, all_mask_flat = [], []

        with torch.no_grad():
            for imgs, masks in test_dl:
                imgs, masks = imgs.to(device), masks.to(device)
                with autocast('cuda', enabled=CFG['use_amp']):
                    logits = model(imgs); loss = criterion(logits, masks)
                test_loss += loss.item()
                probs = torch.sigmoid(logits.float()).squeeze(1).cpu().numpy()
                m_np  = masks.squeeze(1).cpu().numpy()
                all_probs_flat.extend(probs.ravel())
                all_mask_flat.extend(m_np.ravel())
        test_loss /= len(test_dl)

        all_probs_flat = np.array(all_probs_flat)
        all_mask_flat  = np.array(all_mask_flat, dtype=np.int32)
        all_preds_flat = (all_probs_flat >= CFG['thr_pixel']).astype(np.int32)

        f1_px = f1_score(all_mask_flat, all_preds_flat, zero_division=0)
        tp    = (all_preds_flat * all_mask_flat).sum()
        dice  = (2*tp) / (all_preds_flat.sum() + all_mask_flat.sum() + 1e-8)
        iou   = jaccard_score(all_mask_flat, all_preds_flat, zero_division=0)

        history['train_loss'].append(train_loss); history['val_loss'].append(test_loss)
        history['val_f1_pixel'].append(f1_px); history['val_dice'].append(float(dice))
        history['val_iou'].append(float(iou))

        if f1_px > best_f1:
            best_f1, best_epoch, no_improve = f1_px, epoch, 0
            torch.save(model.state_dict(),
                       output_dir / f'region{rid}_{name}_best.pt')
        else:
            no_improve += 1

        print(f'  Ep {epoch:2d}/{CFG["epochs"]} | '
              f'loss={train_loss:.4f}/{test_loss:.4f} | '
              f'F1={f1_px:.4f} | Dice={dice:.4f} | IoU={iou:.4f}'
              + (' ★' if no_improve == 0 else ''))
        if no_improve >= CFG['patience']:
            print(f'  ⏹  Early stopping en época {epoch}'); break

    # ── Evaluación final con mejor checkpoint ────────────────────────────────
    model.load_state_dict(torch.load(
        output_dir / f'region{rid}_{name}_best.pt', map_location=device))
    model.eval()
    all_probs_flat, all_mask_flat = [], []
    vis_imgs, vis_masks, vis_preds = None, None, None

    with torch.no_grad():
        for batch_i, (imgs, masks) in enumerate(test_dl):
            imgs_dev = imgs.to(device)
            with autocast('cuda', enabled=CFG['use_amp']):
                logits = model(imgs_dev)
            probs = torch.sigmoid(logits.float()).squeeze(1).cpu().numpy()
            m_np  = masks.squeeze(1).numpy()
            all_probs_flat.extend(probs.ravel())
            all_mask_flat.extend(m_np.ravel())
            if batch_i == 0:
                vis_imgs  = imgs.numpy()[:4]
                vis_masks = m_np[:4]
                vis_preds = probs[:4]

    all_probs_flat = np.array(all_probs_flat)
    all_mask_flat  = np.array(all_mask_flat, dtype=np.int32)
    all_preds_flat = (all_probs_flat >= CFG['thr_pixel']).astype(np.int32)

    # Umbral óptimo
    prec_c, rec_c, thr_c = precision_recall_curve(all_mask_flat, all_probs_flat)
    f1_curve  = 2 * prec_c * rec_c / (prec_c + rec_c + 1e-8)
    best_thr  = float(thr_c[np.argmax(f1_curve[:-1])])
    preds_opt = (all_probs_flat >= best_thr).astype(np.int32)

    tp05 = (all_preds_flat * all_mask_flat).sum()
    tp_o = (preds_opt * all_mask_flat).sum()

    region_result = {
        'region_id':        rid,
        'region_name':      info['name'],
        'country':          info['country'],
        'trigger':          info['trigger'],
        'train_regions':    train_regions,
        'n_train':          int(len(train_idx)),
        'n_test':           int(len(test_idx)),
        'best_epoch':       best_epoch,
        'best_thr_pixel':   best_thr,
        'f1_pixel_thr05':   float(f1_score(all_mask_flat, all_preds_flat, zero_division=0)),
        'f1_pixel_thr_opt': float(f1_score(all_mask_flat, preds_opt,      zero_division=0)),
        'dice_thr05':       float(2*tp05 / (all_preds_flat.sum() + all_mask_flat.sum() + 1e-8)),
        'dice_thr_opt':     float(2*tp_o  / (preds_opt.sum()    + all_mask_flat.sum() + 1e-8)),
        'iou_thr05':        float(jaccard_score(all_mask_flat, all_preds_flat, zero_division=0)),
        'iou_thr_opt':      float(jaccard_score(all_mask_flat, preds_opt,      zero_division=0)),
        'auc_roc':          float(roc_auc_score(all_mask_flat, all_probs_flat)),
        'auc_pr':           float(average_precision_score(all_mask_flat, all_probs_flat)),
        'history':          history,
    }

    with open(output_dir / f'region{rid}_{name}_results.json', 'w') as f:
        json.dump(region_result, f, indent=2)

    if vis_imgs is not None:
        np.savez_compressed(
            output_dir / f'region{rid}_{name}_vis.npz',
            imgs=vis_imgs, masks=vis_masks, preds=vis_preds)

    elapsed = (time.time() - t_region) / 60
    print(f'\n  ✅ {info["name"]} GUARDADO | '
          f'F1@0.5={region_result["f1_pixel_thr05"]:.4f} | '
          f'F1@opt={region_result["f1_pixel_thr_opt"]:.4f} | '
          f'Dice={region_result["dice_thr05"]:.4f} | '
          f'IoU={region_result["iou_thr05"]:.4f} | {elapsed:.1f}min')

    del model; torch.cuda.empty_cache()

# ── Resumen final de sesión ────────────────────────────────────────────────
done_now    = [r for r in range(4) if region_is_done(r)]
pending_now = [r for r in range(4) if not region_is_done(r)]
print(f'\nRegiones completadas: {[REGION_RANGES[r]["name"] for r in done_now]}')
if pending_now:
    print(f'Regiones pendientes:  {[REGION_RANGES[r]["name"] for r in pending_now]}')
    print('→ Inicia una nueva sesión y vuelve a correr desde la Celda 1')
else:
    print('🎉 ¡Todas las regiones completadas! Ejecuta las celdas de análisis.')

## Análisis de Resultados LORO
Las celdas siguientes cargan los resultados desde Drive y pueden ejecutarse en cualquier sesión, incluso sin GPU, siempre que los JSON estén guardados.

In [ ]:
# ── Celda 7: Carga de resultados y tabla comparativa ──────────────────────
# (puede correrse sin GPU, solo necesita los JSON en Drive)

# Montar Drive si no está montado
import os
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
    from pathlib import Path
    import json, numpy as np
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    base_path  = Path('/content/drive/MyDrive/Landslide4Sense')
    output_dir = base_path / 'results' / 'comparable_literatura' / 'loro_cross_region'
    REGION_RANGES = {
        0: {'name': 'Iburi',  'country': 'Japón',  'trigger': 'Sísmico'},
        1: {'name': 'Kodagu', 'country': 'India',  'trigger': 'Lluvia'},
        2: {'name': 'Gorkha', 'country': 'Nepal',  'trigger': 'Sísmico'},
        3: {'name': 'Taiwan', 'country': 'Taiwán', 'trigger': 'Tifón/Lluvia'},
    }

# Cargar todos los resultados disponibles
loro_results = {}
for rid, info in REGION_RANGES.items():
    name = info['name'].lower()
    path = output_dir / f'region{rid}_{name}_results.json'
    if path.exists():
        with open(path) as f:
            loro_results[rid] = json.load(f)
    else:
        print(f'⚠️  No encontrado: {path.name}')

if not loro_results:
    print('❌ No hay resultados LORO. Corre primero las celdas de entrenamiento.')
else:
    print(f'✅ {len(loro_results)}/4 regiones cargadas\n')

    # Tabla de resultados
    COLS = ['F1@0.5', 'F1@opt', 'Dice', 'IoU', 'AUC-ROC', 'AUC-PR', 'Época']
    header = f'{"Región":8} {"País":7} {"Detonante":13} ' + ' '.join(f'{c:>8}' for c in COLS)
    print('='*len(header))
    print(header)
    print('='*len(header))

    f1_vals, dice_vals, iou_vals, auc_vals = [], [], [], []
    for rid in range(4):
        if rid not in loro_results: continue
        r = loro_results[rid]
        vals = [r['f1_pixel_thr05'], r['f1_pixel_thr_opt'], r['dice_thr05'],
                r['iou_thr05'], r['auc_roc'], r['auc_pr'], r['best_epoch']]
        f1_vals.append(r['f1_pixel_thr05'])
        dice_vals.append(r['dice_thr05'])
        iou_vals.append(r['iou_thr05'])
        auc_vals.append(r['auc_roc'])
        row = f'{r["region_name"]:8} {r["country"]:7} {r["trigger"]:13} '
        row += ' '.join(f'{v:>8.4f}' if isinstance(v, float) else f'{v:>8d}' for v in vals)
        print(row)

    if len(f1_vals) == 4:
        print('='*len(header))
        print(f'{"MEDIA":8} {"":7} {"":13} '
              f'{np.mean(f1_vals):>8.4f} {"":>8} {np.mean(dice_vals):>8.4f} '
              f'{np.mean(iou_vals):>8.4f} {np.mean(auc_vals):>8.4f}')
        print(f'{"STD":8} {"":7} {"":13} '
              f'{np.std(f1_vals):>8.4f} {"":>8} {np.std(dice_vals):>8.4f} '
              f'{np.std(iou_vals):>8.4f} {np.std(auc_vals):>8.4f}')

    # Guardar resumen LORO
    loro_summary = {
        'protocol': 'LORO',
        'model': 'UNet-ResNet34',
        'n_regions': len(loro_results),
        'mean_f1_pixel': float(np.mean(f1_vals)) if f1_vals else None,
        'std_f1_pixel':  float(np.std(f1_vals))  if f1_vals else None,
        'mean_dice':     float(np.mean(dice_vals)) if dice_vals else None,
        'mean_iou':      float(np.mean(iou_vals))  if iou_vals else None,
        'mean_auc_roc':  float(np.mean(auc_vals))  if auc_vals else None,
        'per_region': {REGION_RANGES[rid]['name']: {
                'f1_pixel': r['f1_pixel_thr05'], 'dice': r['dice_thr05'],
                'iou': r['iou_thr05'], 'auc_roc': r['auc_roc']
            } for rid, r in loro_results.items()
        }
    }
    with open(output_dir / 'loro_summary.json', 'w') as f:
        json.dump(loro_summary, f, indent=2)
    print(f'\n💾 Resumen guardado: {output_dir / "loro_summary.json"}')

In [ ]:
# ── Celda 8: Visualización de predicciones por región ─────────────────────

def error_map_rgb(mask, pred_prob, thr=0.5):
    """Mapa de error: TP=verde, FP=rojo, FN=naranja, TN=negro."""
    pred = (pred_prob >= thr).astype(np.float32)
    h, w = mask.shape
    rgb = np.zeros((h, w, 3), dtype=np.float32)
    tp = (pred == 1) & (mask == 1)   # verde
    fp = (pred == 1) & (mask == 0)   # rojo
    fn = (pred == 0) & (mask == 1)   # naranja
    rgb[tp] = [0.20, 0.80, 0.20]
    rgb[fp] = [0.90, 0.10, 0.10]
    rgb[fn] = [1.00, 0.60, 0.00]
    return rgb

regions_with_vis = [
    rid for rid in range(4)
    if (output_dir / f'region{rid}_{REGION_RANGES[rid]["name"].lower()}_vis.npz').exists()
]

if not regions_with_vis:
    print('⚠️  No se encontraron archivos _vis.npz. Entrena al menos una región primero.')
else:
    for rid in regions_with_vis:
        info = REGION_RANGES[rid]
        name = info['name'].lower()
        vis  = np.load(output_dir / f'region{rid}_{name}_vis.npz')
        imgs  = vis['imgs'];  masks = vis['masks'];  preds = vis['preds']
        n_show = min(4, len(imgs))

        fig, axes = plt.subplots(n_show, 4, figsize=(16, 4*n_show))
        fig.suptitle(f'LORO — Held-out: {info["name"]} ({info["country"]}) | {info["trigger"]}',
                     fontsize=14, fontweight='bold')

        for i in range(n_show):
            ax = axes[i] if n_show > 1 else axes
            # RGB (canales 2,1,0 = B4,B3,B2)
            rgb = imgs[i][[2,1,0]].transpose(1,2,0)
            for c in range(3):
                ch = rgb[:,:,c]; p1, p99 = np.percentile(ch,[1,99])
                rgb[:,:,c] = np.clip((ch-p1)/(p99-p1+1e-8), 0, 1)

            ax[0].imshow(rgb); ax[0].set_title('RGB (B4-B3-B2)'); ax[0].axis('off')
            ax[1].imshow(masks[i], cmap='gray', vmin=0, vmax=1)
            ax[1].set_title('GT Máscara'); ax[1].axis('off')
            ax[2].imshow(preds[i], cmap='RdYlGn', vmin=0, vmax=1)
            ax[2].set_title('Probabilidad'); ax[2].axis('off')
            ax[3].imshow(error_map_rgb(masks[i], preds[i]))
            ax[3].set_title('Mapa de Error'); ax[3].axis('off')

        # Leyenda mapa de error
        legend_elems = [
            mpatches.Patch(color=(0.2,0.8,0.2), label='TP'),
            mpatches.Patch(color=(0.9,0.1,0.1), label='FP'),
            mpatches.Patch(color=(1.0,0.6,0.0), label='FN'),
            mpatches.Patch(color=(0.0,0.0,0.0), label='TN'),
        ]
        fig.legend(handles=legend_elems, loc='lower center', ncol=4, fontsize=10)
        plt.tight_layout(rect=[0,0.03,1,1])
        save_path = output_dir / f'region{rid}_{name}_predictions.png'
        plt.savefig(save_path, dpi=120, bbox_inches='tight')
        plt.show()
        print(f'💾 {save_path.name}')

In [ ]:
# ── Celda 9: LORO vs 5-Fold CV — Gap de generalización geográfica ──────────
#
# Carga los resultados de L4S_04 (5-fold CV) para comparación directa.
# Necesita que L4S_04 haya corrido y guardado fold*_results.json en Drive.

fivefold_dir = base_path / 'results' / 'comparable_literatura' / 'unet_5fold'

fivefold_f1s = []
for fold in range(1, 6):
    p = fivefold_dir / f'fold{fold}_results.json'
    if p.exists():
        with open(p) as f:
            fivefold_f1s.append(json.load(f)['f1_pixel_thr05'])

loro_f1s = [loro_results[rid]['f1_pixel_thr05'] for rid in sorted(loro_results)]

if not fivefold_f1s:
    print('⚠️  No se encontraron resultados 5-Fold (fold*_results.json) en')
    print(f'   {fivefold_dir}')
    print('   Corre L4S_04 primero. Se mostrará solo el análisis LORO.')
    have_5fold = False
else:
    have_5fold = True
    f1_5fold = np.mean(fivefold_f1s)
    f1_loro  = np.mean(loro_f1s) if loro_f1s else None
    print(f'5-Fold CV F1 (mean ± std): {f1_5fold:.4f} ± {np.std(fivefold_f1s):.4f}  (n={len(fivefold_f1s)})')
    if f1_loro:
        gap = f1_5fold - f1_loro
        print(f'LORO     F1 (mean ± std): {f1_loro:.4f} ± {np.std(loro_f1s):.4f}  (n={len(loro_f1s)})')
        print(f'GAP (5-Fold − LORO):       {gap:+.4f}  '
              f'({"degradación" if gap > 0 else "mejora"} al generalizar)')
        if gap > 0.10:
            print('⚠️  Gap > 0.10 — sesgo de distribución geográfico significativo.')
            print('   Fine-tuning con datos colombianos es ALTAMENTE recomendable.')
        elif gap > 0.05:
            print('⚠️  Gap moderado (0.05–0.10) — considerar fine-tuning con datos colombianos.')
        else:
            print('✅ Gap < 0.05 — el modelo generaliza bien entre regiones.')

# ── Figura 1: Bar chart LORO por región + línea 5-Fold ────────────────────
if loro_results:
    COLORES = {
        0: '#E76F51',  # Iburi/Japón
        1: '#2A9D8F',  # Kodagu/India
        2: '#264653',  # Gorkha/Nepal
        3: '#F4A261',  # Taiwan
    }
    METRICAS = [
        ('f1_pixel_thr05', 'F1-score (thr=0.5)', '#3A86FF'),
        ('dice_thr05',     'Dice',                '#8338EC'),
        ('iou_thr05',      'IoU',                 '#FF006E'),
        ('auc_roc',        'AUC-ROC',             '#FB5607'),
    ]

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('LORO Cross-Region — Generalización Geográfica U-Net (ResNet-34)',
                 fontsize=13, fontweight='bold')

    # Panel izquierdo: métricas por región
    ax = axes[0]
    rids_done = sorted(loro_results.keys())
    x     = np.arange(len(rids_done))
    width = 0.20
    offsets = np.linspace(-(len(METRICAS)-1)/2, (len(METRICAS)-1)/2, len(METRICAS)) * width

    for (key, label, color), offset in zip(METRICAS, offsets):
        vals = [loro_results[rid][key] for rid in rids_done]
        bars = ax.bar(x + offset, vals, width=width, label=label, color=color, alpha=0.85)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                    f'{v:.3f}', ha='center', va='bottom', fontsize=7, rotation=90)

    xlabels = [f'{REGION_RANGES[rid]["name"]}\n({REGION_RANGES[rid]["country"]})'
               for rid in rids_done]
    ax.set_xticks(x); ax.set_xticklabels(xlabels, fontsize=10)
    ax.set_ylim(0, 1.05); ax.set_ylabel('Métrica')
    ax.set_title('Métricas por región held-out')
    ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)

    if have_5fold:
        ax.axhline(f1_5fold, color='black', lw=1.5, ls='--',
                   label=f'5-Fold F1={f1_5fold:.3f}')
        ax.legend(fontsize=8)

    # Panel derecho: comparación LORO vs 5-Fold
    ax2 = axes[1]
    if have_5fold and len(loro_f1s) == 4:
        cats   = [REGION_RANGES[r]['name'] for r in sorted(loro_results)]
        colors = [COLORES[r] for r in sorted(loro_results)]
        bars2  = ax2.bar(cats, loro_f1s, color=colors, alpha=0.85, label='LORO F1')
        for bar, v in zip(bars2, loro_f1s):
            ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                     f'{v:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

        ax2.axhline(f1_5fold, color='#3A86FF', lw=2.0, ls='--',
                    label=f'5-Fold CV: {f1_5fold:.3f}')
        ax2.axhline(np.mean(loro_f1s), color='#FF006E', lw=1.5, ls=':',
                    label=f'LORO media: {np.mean(loro_f1s):.3f}')

        # Zona de gap
        ax2.axhspan(np.mean(loro_f1s), f1_5fold, alpha=0.10, color='gray',
                    label=f'Gap = {f1_5fold - np.mean(loro_f1s):.3f}')

        ax2.set_ylim(0, 1.05); ax2.set_ylabel('F1-score (pixel, thr=0.5)')
        ax2.set_title('F1 LORO vs 5-Fold CV')
        ax2.legend(fontsize=9); ax2.grid(axis='y', alpha=0.3)

        for tick in ax2.get_xticklabels():
            tick.set_fontsize(10)
    else:
        ax2.text(0.5, 0.5,
                 'Completa las 4 regiones y\ncorre L4S_04 para ver el gap.',
                 ha='center', va='center', transform=ax2.transAxes, fontsize=12)
        ax2.set_title('LORO vs 5-Fold (pendiente)')

    plt.tight_layout()
    save_path = output_dir / 'loro_vs_5fold_comparison.png'
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'💾 {save_path}')

In [ ]:
# ── Celda 10: Curvas de convergencia por región ────────────────────────────

if loro_results:
    n_done = len(loro_results)
    fig, axes = plt.subplots(n_done, 3, figsize=(15, 4.5*n_done))
    if n_done == 1:
        axes = axes.reshape(1, -1)
    fig.suptitle('Curvas de Convergencia LORO — U-Net por Región Held-out',
                 fontsize=13, fontweight='bold')

    for row, rid in enumerate(sorted(loro_results)):
        r    = loro_results[rid]
        hist = r['history']
        info = REGION_RANGES[rid]
        eps  = range(1, len(hist['train_loss']) + 1)

        # Loss
        axes[row,0].plot(eps, hist['train_loss'], 'b-',  lw=1.5, label='Train')
        axes[row,0].plot(eps, hist['val_loss'],   'r--', lw=1.5, label=f'Test ({info["name"]})')
        axes[row,0].axvline(r['best_epoch'], color='green', ls=':', lw=1, label=f'Best ep={r["best_epoch"]}')
        axes[row,0].set_title(f'{info["name"]} — Loss'); axes[row,0].legend(fontsize=8)
        axes[row,0].set_xlabel('Época'); axes[row,0].grid(alpha=0.3)

        # F1 + Dice
        axes[row,1].plot(eps, hist['val_f1_pixel'], '#3A86FF', lw=1.5, label='F1 pixel')
        axes[row,1].plot(eps, hist['val_dice'],     '#8338EC', lw=1.5, ls='--', label='Dice')
        axes[row,1].axhline(r['f1_pixel_thr05'], color='#3A86FF', ls=':', lw=1)
        axes[row,1].set_title(f'{info["name"]} — F1 / Dice (test region)')
        axes[row,1].set_ylim(0, 1); axes[row,1].legend(fontsize=8)
        axes[row,1].set_xlabel('Época'); axes[row,1].grid(alpha=0.3)

        # IoU + gap train-test
        axes[row,2].plot(eps, hist['val_iou'], '#FF006E', lw=1.5, label='IoU')
        gap_arr = np.array(hist['train_loss']) - np.array(hist['val_loss'])
        ax_twin = axes[row,2].twinx()
        ax_twin.plot(eps, np.abs(gap_arr), 'gray', lw=1, ls='--', alpha=0.7,
                     label='|Loss gap|')
        ax_twin.set_ylabel('|Gap loss|', fontsize=8, color='gray')
        axes[row,2].set_title(f'{info["name"]} — IoU + Loss gap')
        axes[row,2].set_ylim(0, 1); axes[row,2].legend(fontsize=8, loc='upper left')
        ax_twin.legend(fontsize=8, loc='upper right')
        axes[row,2].set_xlabel('Época'); axes[row,2].grid(alpha=0.3)

    plt.tight_layout()
    save_path = output_dir / 'loro_convergence_curves.png'
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'💾 {save_path}')

In [ ]:
# ── Celda 11: Análisis de dificultad por región + implicaciones Colombia ───

if len(loro_results) == 4:
    print('ANÁLISIS DE DIFICULTAD — ¿Qué región es más difícil de generalizar?')
    print('='*65)

    # Ordenar por F1 ascendente (más difícil primero)
    sorted_regions = sorted(
        [(rid, loro_results[rid]) for rid in loro_results],
        key=lambda x: x[1]['f1_pixel_thr05']
    )

    for rank, (rid, r) in enumerate(sorted_regions):
        info = REGION_RANGES[rid]
        badge = ['🔴 MÁS DIFÍCIL', '🟠', '🟡', '🟢 MÁS FÁCIL'][rank]
        print(f'\n{badge} {r["region_name"]} ({r["country"]}) — Detonante: {r["trigger"]}')
        print(f'   F1={r["f1_pixel_thr05"]:.4f} | Dice={r["dice_thr05"]:.4f} | '
              f'IoU={r["iou_thr05"]:.4f} | AUC-ROC={r["auc_roc"]:.4f}')
        print(f'   Entrenado con: {r["train_regions"]}')

    # Región más similar a Colombia
    print()
    print('='*65)
    print('IMPLICACIONES PARA CONTEXTO COLOMBIANO')
    print('='*65)
    print()
    print('Colombia comparte características con:')
    print('  • Kodagu (India): lluvias tropicales intensas, vegetación densa, suelos')
    print('    meteorizado — detonante LLUVIA similar al contexto Andino colombiano.')
    print('  • Gorkha (Nepal): topografía montañosa compleja, pendientes fuertes,')
    print('    aunque el detonante fue sísmico.')
    print()

    # Identificar región más relevante
    f1_kodagu = loro_results.get(1, {}).get('f1_pixel_thr05', None)
    f1_gorkha = loro_results.get(2, {}).get('f1_pixel_thr05', None)

    if f1_kodagu is not None:
        print(f'  F1 al generalizar a Kodagu (India, lluvia): {f1_kodagu:.4f}')
        print(f'  → Esta es la estimación más conservadora del desempeño esperado')
        print(f'    en Colombia antes de fine-tuning.')

    # Estadísticas de detonante
    trigger_f1 = {'Sísmico': [], 'Lluvia': [], 'Tifón/Lluvia': []}
    for rid, r in loro_results.items():
        trig = REGION_RANGES[rid]['trigger']
        trigger_f1[trig].append(r['f1_pixel_thr05'])

    print()
    print('F1 promedio por tipo de detonante (held-out):')
    for trig, vals in trigger_f1.items():
        if vals:
            print(f'  {trig:15}: {np.mean(vals):.4f}')

    print()
    print('RECOMENDACIÓN:')
    avg_loro_f1 = np.mean([r['f1_pixel_thr05'] for r in loro_results.values()])
    if avg_loro_f1 < 0.50:
        print('  🔴 LORO F1 < 0.50 — Fine-tuning con datos colombianos es ESENCIAL.')
        print('     Explorar: domain adaptation, few-shot learning con 50-100 parches.')
    elif avg_loro_f1 < 0.65:
        print('  🟡 LORO F1 en rango moderado — Fine-tuning recomendado.')
        print('     El modelo puede usarse como inicialización para transfer learning.')
    else:
        print('  🟢 LORO F1 > 0.65 — Buena generalización inter-regional.')
        print('     El modelo puede aplicarse a Colombia con ajuste mínimo.')

else:
    print(f'⚠️  Solo {len(loro_results)}/4 regiones completadas.')
    print('   Completa el entrenamiento para ver el análisis completo.')

In [ ]:
# ── Celda 12: Resumen ejecutivo y próximos pasos ───────────────────────────

print('=' * 65)
print('  RESUMEN EJECUTIVO — LORO Cross-Region Analysis')
print('=' * 65)

# LORO
if loro_results:
    f1s   = [r['f1_pixel_thr05'] for r in loro_results.values()]
    dices = [r['dice_thr05']      for r in loro_results.values()]
    ious  = [r['iou_thr05']       for r in loro_results.values()]
    print(f'\n📍 LORO U-Net — {len(loro_results)}/4 regiones')
    print(f'   F1    (mean ± std): {np.mean(f1s):.4f} ± {np.std(f1s):.4f}')
    print(f'   Dice  (mean ± std): {np.mean(dices):.4f} ± {np.std(dices):.4f}')
    print(f'   IoU   (mean ± std): {np.mean(ious):.4f} ± {np.std(ious):.4f}')
    worst = min(loro_results.items(), key=lambda x: x[1]['f1_pixel_thr05'])
    best  = max(loro_results.items(), key=lambda x: x[1]['f1_pixel_thr05'])
    print(f'   Mejor región:  {REGION_RANGES[best[0]]["name"]:7} F1={best[1]["f1_pixel_thr05"]:.4f}')
    print(f'   Peor región:   {REGION_RANGES[worst[0]]["name"]:7} F1={worst[1]["f1_pixel_thr05"]:.4f}')

# Comparación con 5-Fold
if have_5fold and loro_results:
    f1_l = np.mean([r['f1_pixel_thr05'] for r in loro_results.values()])
    print(f'\n📊 COMPARACIÓN:')
    print(f'   5-Fold CV  F1: {f1_5fold:.4f}  (rendimiento in-distribution)')
    print(f'   LORO       F1: {f1_l:.4f}  (rendimiento cross-region)')
    print(f'   GAP:           {f1_5fold - f1_l:+.4f}')

print(f'\n💾 Archivos guardados en Drive:')
print(f'   {output_dir}')
for rid in sorted(loro_results):
    name = REGION_RANGES[rid]['name'].lower()
    print(f'   → region{rid}_{name}_best.pt + _results.json + _vis.npz')
print(f'   → loro_summary.json')
print(f'   → loro_vs_5fold_comparison.png')
print(f'   → loro_convergence_curves.png')

print(f'\n🔜 PRÓXIMOS PASOS:')
print(f'   1. Revisión L4S_08 — incorporar métricas LORO en estado del arte')
print(f'   2. Recolección de datos colombianos (imágenes + máscaras)')
print(f'   3. Fine-tuning del modelo LORO con parches colombianos (transfer learning)')
print(f'   4. Análisis de domain gap espectral (Colombia vs L4S regions)')
print(f'   5. Reorganización del repositorio en carpetas temáticas')